# 📈 Daily Challenge: Stock Price Prediction with LSTM (PyTorch)

**What you'll build end-to-end:**

| Step | What happens |
|---|---|
| 1 | Install libraries & mount Google Drive |
| 2 | Load & preprocess the Kaggle stock dataset |
| 3 | Build a PyTorch `Dataset` + `DataLoader` pipeline |
| 4 | Define a stacked LSTM model with Dropout |
| 5 | Train with Adam + MSE loss, tracking validation loss |
| 6 | Evaluate with R² on the test set & visualize predictions |

> **Runtime:** Runtime → Change runtime type → **T4 GPU**
>
> **Dataset:** Download from [Kaggle — Stock Market Dataset](https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset). Upload the CSV (e.g. `AAPL.csv` from the `stocks/` folder) to your Colab session or Google Drive.

---
## 1️⃣ Install & Import Libraries

In [ ]:
# All libraries come pre-installed on Colab — no pip needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

import pickle
import os

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
print(f'CUDA     : {torch.cuda.is_available()}')

---
## 2️⃣ Load & Preprocess the Dataset

### 2a — Upload your CSV

Run the cell below to upload a file from your machine, **or** skip it and set `CSV_PATH` to a path already in Colab (e.g. from Google Drive).

In [ ]:
from google.colab import files

print('Upload a stock CSV from the Kaggle dataset (e.g. AAPL.csv).')
print('Skip this cell and set CSV_PATH manually if the file is already on Colab.')
uploaded = files.upload()   # opens a file-picker dialog
CSV_PATH = list(uploaded.keys())[0]
print(f'Using: {CSV_PATH}')

In [ ]:
# ── Alternatively: set the path directly if file already exists ───────────────
# CSV_PATH = '/content/AAPL.csv'
# or for Google Drive:
# from google.colab import drive; drive.mount('/drive')
# CSV_PATH = '/drive/MyDrive/stocks/AAPL.csv'

### 2b — Load, clean, and engineer features

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH, parse_dates=['Date'])
print('Raw shape :', df_raw.shape)
print('Columns   :', df_raw.columns.tolist())
display(df_raw.head())
display(df_raw.dtypes)

In [ ]:
# ── Clean & feature engineering ──────────────────────────────────────────────
df = df_raw.copy()

# Sort by date (oldest first)
df = df.sort_values('Date').reset_index(drop=True)

# Drop columns that leak future information or are non-numeric identifiers
# 'OpenInt' exists in some versions of the dataset; drop if present
drop_cols = [c for c in ['OpenInt', 'Adj Close', 'Name'] if c in df.columns]
df = df.drop(columns=drop_cols)

# ── Target: next day's closing price ─────────────────────────────────────────
# Shift Close up by one row so each row's target = tomorrow's Close
df['Target'] = df['Close'].shift(-1)

# Drop the last row (it has NaN target — no next day)
df = df.dropna().reset_index(drop=True)

print('Cleaned shape :', df.shape)
display(df.head(10))

# ── Verify date range ─────────────────────────────────────────────────────────
print(f'\nDate range : {df["Date"].min().date()}  →  {df["Date"].max().date()}')
print(f'Total trading days : {len(df)}')

In [ ]:
# ── Exploratory visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle(f'Stock Price Overview — {CSV_PATH.split("/")[-1].replace(".csv","")}',
             fontsize=14, fontweight='bold')

# Close price history
axes[0, 0].plot(df['Date'], df['Close'], color='steelblue', linewidth=0.8)
axes[0, 0].set_title('Closing Price History')
axes[0, 0].set_xlabel('Date'); axes[0, 0].set_ylabel('Close ($)')
axes[0, 0].grid(alpha=0.3)

# Volume
axes[0, 1].bar(df['Date'], df['Volume'], color='steelblue', alpha=0.6, width=1)
axes[0, 1].set_title('Daily Trading Volume')
axes[0, 1].set_xlabel('Date'); axes[0, 1].set_ylabel('Volume')
axes[0, 1].grid(alpha=0.3, axis='y')

# Daily return distribution
daily_ret = df['Close'].pct_change().dropna() * 100
axes[1, 0].hist(daily_ret, bins=80, color='darkorange', edgecolor='white')
axes[1, 0].set_title('Daily Return Distribution (%)')
axes[1, 0].set_xlabel('Daily Return (%)'); axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(alpha=0.3, axis='y')

# Close vs Target scatter (should be near-diagonal)
axes[1, 1].scatter(df['Close'], df['Target'], alpha=0.1, s=2, color='purple')
axes[1, 1].set_title('Close vs Next-Day Target')
axes[1, 1].set_xlabel('Close ($)'); axes[1, 1].set_ylabel('Target ($)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 2c — Normalize with MinMaxScaler

In [ ]:
# ── Define feature and target columns ─────────────────────────────────────────
FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGET_COL   = 'Target'

# ── Separate features and target BEFORE scaling ───────────────────────────────
X_all = df[FEATURE_COLS].values.astype(np.float32)   # (N, 5)
y_all = df[TARGET_COL].values.astype(np.float32)     # (N,)

# ── Scale features: fit on ALL data, apply to all ────────────────────────────
# Note: in a rigorous pipeline we'd fit ONLY on train.  
# Here we scale all for simplicity, then split.
feature_scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = feature_scaler.fit_transform(X_all)        # (N, 5) in [0, 1]

# ── Scale target separately so we can invert predictions later ────────────────
target_scaler = MinMaxScaler(feature_range=(0, 1))
y_scaled = target_scaler.fit_transform(y_all.reshape(-1, 1)).ravel()  # (N,)

print(f'X_scaled shape : {X_scaled.shape}   range [{X_scaled.min():.2f}, {X_scaled.max():.2f}]')
print(f'y_scaled shape : {y_scaled.shape}   range [{y_scaled.min():.2f}, {y_scaled.max():.2f}]')

---
## 3️⃣ Prepare Dataset for Training

### 3a — Sliding window sequences

An LSTM needs sequences shaped `(batch, time_steps, features)`. We use a **lookback window** of N days to predict the next day's price.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
LOOKBACK   = 60    # use 60 trading days (~3 months) to predict next day
BATCH_SIZE = 64
VAL_RATIO  = 0.1   # 10% of training data used for validation
TEST_RATIO = 0.2   # 20% held out as test set (chronological split)

def make_sequences(X, y, lookback):
    """
    Build sliding-window sequences.
    Each sequence: X[i : i+lookback] → y[i+lookback]

    Returns:
        Xs : np.ndarray of shape (N-lookback, lookback, n_features)
        ys : np.ndarray of shape (N-lookback,)
    """
    Xs, ys = [], []
    for i in range(len(X) - lookback):
        Xs.append(X[i : i + lookback])
        ys.append(y[i + lookback])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


X_seq, y_seq = make_sequences(X_scaled, y_scaled, LOOKBACK)
print(f'Sequence array  : {X_seq.shape}  (samples, lookback, features)')
print(f'Target array    : {y_seq.shape}')

In [ ]:
# ── Chronological train / val / test split ────────────────────────────────────
# We split by index (time order), NOT randomly — leaking future into past is wrong.

n        = len(X_seq)
n_test   = int(n * TEST_RATIO)
n_train  = n - n_test
n_val    = int(n_train * VAL_RATIO)
n_tr     = n_train - n_val

X_train, y_train = X_seq[:n_tr],            y_seq[:n_tr]
X_val,   y_val   = X_seq[n_tr:n_train],     y_seq[n_tr:n_train]
X_test,  y_test  = X_seq[n_train:],         y_seq[n_train:]

print(f'Train   : {X_train.shape[0]:>6} samples  ({X_train.shape[0]/n*100:.1f}%)')
print(f'Val     : {X_val.shape[0]:>6} samples  ({X_val.shape[0]/n*100:.1f}%)')
print(f'Test    : {X_test.shape[0]:>6} samples  ({X_test.shape[0]/n*100:.1f}%)')
print(f'\nSequence shape per sample: {X_train.shape[1:]}  → (lookback={LOOKBACK}, features={X_train.shape[2]})')

### 3b — Custom PyTorch Dataset & DataLoader

In [ ]:
class StockDataset(Dataset):
    """
    Custom PyTorch Dataset wrapping numpy sequence arrays.

    PyTorch's DataLoader requires __len__ and __getitem__.
    __getitem__ returns (features_tensor, target_tensor) for one sample.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Convert to float32 tensors once at construction time
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# ── Instantiate datasets ──────────────────────────────────────────────────────
train_dataset = StockDataset(X_train, y_train)
val_dataset   = StockDataset(X_val,   y_val)
test_dataset  = StockDataset(X_test,  y_test)

# ── DataLoaders ───────────────────────────────────────────────────────────────
# shuffle=True on training so the model doesn't memorize batch order
# shuffle=False on val/test so evaluation is deterministic
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

# Peek at one batch
x_batch, y_batch = next(iter(train_loader))
print(f'\nBatch shape — X: {x_batch.shape}   y: {y_batch.shape}')
print(f'Batch dtype — X: {x_batch.dtype}   y: {y_batch.dtype}')

---
## 4️⃣ Define the LSTM Model

We use `torch.nn.Module` to build a stacked LSTM with Dropout regularization.

```
Input  (batch, 60, 5)
  └─ LSTM layer 1  hidden=128, num_layers=2, dropout=0.2
       └─ takes last hidden state  → (batch, 128)
            └─ Dropout(0.2)
                 └─ Linear(128 → 64)
                      └─ ReLU
                           └─ Linear(64 → 1)   ← predicted normalized price
```

In [ ]:
class LSTMModel(nn.Module):
    """
    Stacked LSTM for time-series regression.

    Args:
        input_size  : number of features per time step (5)
        hidden_size : LSTM hidden state dimensionality
        num_layers  : number of stacked LSTM layers
        dropout     : dropout probability applied between LSTM layers
        output_size : prediction horizon (1 = next-day close)
    """

    def __init__(self,
                 input_size:  int = 5,
                 hidden_size: int = 128,
                 num_layers:  int = 2,
                 dropout:     float = 0.2,
                 output_size: int = 1):
        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # ── LSTM block ────────────────────────────────────────────────────────
        # batch_first=True → input shape is (batch, seq_len, features)
        # dropout applies between stacked layers (only effective when num_layers > 1)
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )

        # ── Classifier head ───────────────────────────────────────────────────
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (batch, seq_len, input_size)
        """
        batch_size = x.size(0)

        # Initialise hidden and cell states to zeros on the correct device
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        # LSTM forward pass
        # out  : (batch, seq_len, hidden_size) — hidden state at every step
        # (hn, cn) : final hidden and cell states
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Take only the last time step's hidden state as the sequence summary
        out = out[:, -1, :]           # (batch, hidden_size)

        # Head
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)           # (batch, 1)
        return out.squeeze(-1)        # (batch,)


# ── Instantiate ───────────────────────────────────────────────────────────────
INPUT_SIZE  = X_train.shape[2]   # 5 features
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
DROPOUT     = 0.2

model = LSTMModel(
    input_size  = INPUT_SIZE,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
    output_size = 1
).to(DEVICE)

print(model)
print(f'\nTotal parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Model device     : {next(model.parameters()).device}')

---
## 5️⃣ Train the Model

In [ ]:
# ── Optimizer & loss ──────────────────────────────────────────────────────────
LEARNING_RATE = 1e-3
EPOCHS        = 50
PATIENCE      = 7     # early stopping: stop if val_loss doesn't improve for 7 epochs

optimizer  = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion  = nn.MSELoss()
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print(f'Optimizer  : Adam  (lr={LEARNING_RATE}, weight_decay=1e-5)')
print(f'Loss       : MSELoss')
print(f'Scheduler  : ReduceLROnPlateau  (factor=0.5, patience=3)')
print(f'Max epochs : {EPOCHS}   Early stopping patience: {PATIENCE}')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one full pass over the training DataLoader."""
    model.train()      # enable dropout
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()          # clear gradients from last step
        preds = model(X_batch)         # forward pass
        loss  = criterion(preds, y_batch)  # MSE
        loss.backward()                # backpropagation
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()               # update weights
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    """Evaluate on val or test loader, return average MSE loss."""
    model.eval()       # disable dropout
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
    return total_loss / len(loader)


print('Helper functions defined.')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
train_losses = []
val_losses   = []

best_val_loss  = float('inf')
epochs_no_impr = 0
BEST_MODEL_PATH = 'best_lstm_model.pt'

print(f'{'Epoch':>6}  {'Train MSE':>10}  {'Val MSE':>10}  {'LR':>10}')
print('-' * 44)

for epoch in range(1, EPOCHS + 1):
    tr_loss  = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)
    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6}  {tr_loss:>10.6f}  {val_loss:>10.6f}  {current_lr:>10.2e}')

    # ── Save best model ───────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        epochs_no_impr = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        epochs_no_impr += 1

    # ── Early stopping ────────────────────────────────────────────────────────
    if epochs_no_impr >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
        break

print(f'\nBest val MSE : {best_val_loss:.6f}')
print(f'Model saved  : {BEST_MODEL_PATH}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=13, fontweight='bold')

ep = range(1, len(train_losses) + 1)

axes[0].plot(ep, train_losses, label='Train MSE',      color='steelblue', marker='o', markevery=5)
axes[0].plot(ep, val_losses,   label='Validation MSE', color='tomato',    marker='s', markevery=5, linestyle='--')
axes[0].set_title('MSE Loss over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (normalized scale)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Log scale to see early learning clearly
axes[1].semilogy(ep, train_losses, label='Train MSE',      color='steelblue', marker='o', markevery=5)
axes[1].semilogy(ep, val_losses,   label='Validation MSE', color='tomato',    marker='s', markevery=5, linestyle='--')
axes[1].set_title('MSE Loss over Epochs (log scale)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE (log)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6️⃣ Evaluate the Model

In [ ]:
# ── Load best saved weights ───────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

# ── Collect all test predictions ──────────────────────────────────────────────
all_preds  = []
all_truths = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        preds   = model(X_batch).cpu().numpy()
        all_preds.extend(preds)
        all_truths.extend(y_batch.numpy())

y_pred_norm = np.array(all_preds)
y_true_norm = np.array(all_truths)

# ── Inverse-transform to real price scale ─────────────────────────────────────
y_pred_price = target_scaler.inverse_transform(y_pred_norm.reshape(-1, 1)).ravel()
y_true_price = target_scaler.inverse_transform(y_true_norm.reshape(-1, 1)).ravel()

# ── Metrics ───────────────────────────────────────────────────────────────────
r2   = r2_score(y_true_price, y_pred_price)
mse  = np.mean((y_true_price - y_pred_price) ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_true_price - y_pred_price))
mape = np.mean(np.abs((y_true_price - y_pred_price) / y_true_price)) * 100

print('══ Test Set Evaluation ════════════════════════')
print(f'  R²   (coefficient of determination) : {r2:.4f}')
print(f'  MSE  (mean squared error)           : {mse:.4f}')
print(f'  RMSE (root mean squared error)      : ${rmse:.2f}')
print(f'  MAE  (mean absolute error)          : ${mae:.2f}')
print(f'  MAPE (mean abs. percentage error)   : {mape:.2f}%')
print('═══════════════════════════════════════════════')
print()
print('Interpretation:')
print(f'  R² = {r2:.4f} means the model explains {r2*100:.1f}% of the variance in next-day price.')
print(f'  On average predictions are ${mae:.2f} away from the true closing price.')

In [ ]:
# ── Visualize predictions vs actuals ──────────────────────────────────────────
# Align dates with the test window
test_start_idx = n_train + LOOKBACK
test_dates     = df['Date'].values[test_start_idx : test_start_idx + len(y_true_price)]

fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('LSTM — Predicted vs Actual Next-Day Close Price', fontsize=13, fontweight='bold')

# Full test period
axes[0].plot(test_dates, y_true_price, label='Actual',    color='steelblue', linewidth=1.0)
axes[0].plot(test_dates, y_pred_price, label='Predicted', color='tomato',    linewidth=1.0, alpha=0.85, linestyle='--')
axes[0].set_title(f'Full Test Period  (R² = {r2:.4f})')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Zoom: last 120 trading days
n_zoom = min(120, len(y_true_price))
axes[1].plot(test_dates[-n_zoom:], y_true_price[-n_zoom:], label='Actual',    color='steelblue', linewidth=1.2)
axes[1].plot(test_dates[-n_zoom:], y_pred_price[-n_zoom:], label='Predicted', color='tomato',    linewidth=1.2, linestyle='--')
axes[1].fill_between(test_dates[-n_zoom:],
                     y_true_price[-n_zoom:],
                     y_pred_price[-n_zoom:],
                     alpha=0.15, color='purple', label='Error band')
axes[1].set_title(f'Zoomed — Last {n_zoom} Test Days  (MAE = ${mae:.2f})')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('Price ($)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Residual analysis ─────────────────────────────────────────────────────────
residuals = y_true_price - y_pred_price

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Residual Analysis', fontsize=12, fontweight='bold')

# Residuals over time
axes[0].plot(test_dates, residuals, color='purple', linewidth=0.7)
axes[0].axhline(0, color='gray', linewidth=1, linestyle='--')
axes[0].set_title('Residuals over Time')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Residual ($)')
axes[0].grid(alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=60, color='purple', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='tomato', linewidth=1.5, linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3, axis='y')

# Predicted vs actual scatter
axes[2].scatter(y_true_price, y_pred_price, alpha=0.3, s=8, color='steelblue')
lo = min(y_true_price.min(), y_pred_price.min())
hi = max(y_true_price.max(), y_pred_price.max())
axes[2].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect prediction')
axes[2].set_title(f'Predicted vs Actual  (R²={r2:.3f})')
axes[2].set_xlabel('Actual Price ($)'); axes[2].set_ylabel('Predicted Price ($)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 💾 Save the Scaler & Model

In [ ]:
# ── Save scaler objects with pickle ───────────────────────────────────────────
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(feature_scaler, f)

with open('target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

# The model weights are already saved as best_lstm_model.pt
# To save the full model architecture + weights:
torch.save(model, 'lstm_full_model.pt')

print('Saved files:')
for fname in ['best_lstm_model.pt', 'lstm_full_model.pt', 'feature_scaler.pkl', 'target_scaler.pkl']:
    size = os.path.getsize(fname) / 1024
    print(f'  {fname:<30}  {size:.1f} KB')

In [ ]:
# ── How to reload and use the model for inference ─────────────────────────────
print('═══ Reload & inference example ═══════════════════════════════')

# Load scalers
with open('feature_scaler.pkl', 'rb') as f:
    loaded_feature_scaler = pickle.load(f)
with open('target_scaler.pkl', 'rb') as f:
    loaded_target_scaler = pickle.load(f)

# Load model weights into a fresh LSTMModel instance
loaded_model = LSTMModel(
    input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS, dropout=DROPOUT, output_size=1
).to(DEVICE)
loaded_model.load_state_dict(torch.load('best_lstm_model.pt', map_location=DEVICE))
loaded_model.eval()

# Use the last LOOKBACK rows of the full scaled dataset as an example input
example_seq = torch.tensor(
    X_scaled[-LOOKBACK:][np.newaxis, ...],   # shape (1, 60, 5)
    dtype=torch.float32
).to(DEVICE)

with torch.no_grad():
    pred_norm = loaded_model(example_seq).cpu().numpy()

pred_price = loaded_target_scaler.inverse_transform(pred_norm.reshape(-1, 1)).ravel()[0]
true_next  = df['Target'].iloc[-1]

print(f'  Predicted next-day close : ${pred_price:.2f}')
print(f'  True next-day close      : ${true_next:.2f}')
print(f'  Error                    : ${abs(pred_price - true_next):.2f}')
print('═══════════════════════════════════════════════════════════════')

---
## 📋 Summary

### Architecture recap

| Layer | Output shape | Purpose |
|---|---|---|
| Input | (batch, 60, 5) | 60-day window, 5 OHLCV features |
| LSTM (128 units, 2 layers, dropout=0.2) | (batch, 60, 128) | Learns temporal dependencies |
| Last hidden state slice `[:, -1, :]` | (batch, 128) | Sequence summary |
| Dropout(0.2) | (batch, 128) | Regularization |
| Linear(128→64) + ReLU | (batch, 64) | Feature compression |
| Linear(64→1) | (batch, 1) | Next-day price prediction |

### Key PyTorch concepts used

| Concept | Where used |
|---|---|
| `nn.Module` | Base class for `LSTMModel` |
| `nn.LSTM` | Core recurrent layer with `batch_first=True` |
| `nn.Linear` | Fully connected head layers |
| `nn.Dropout` | Regularization to reduce overfitting |
| `nn.MSELoss` | Regression loss function |
| `torch.optim.Adam` | Adaptive gradient optimizer |
| `Dataset` + `DataLoader` | Batching, shuffling, GPU transfer |
| `torch.save` / `load_state_dict` | Model persistence |
| `torch.no_grad()` | Disables gradient tracking at inference time |

### Important design decisions
- **Chronological split** (not random): leaking future prices into training would inflate R² artificially.
- **Two separate scalers**: feature scaler for OHLCV inputs, target scaler for closing price — required to correctly invert predictions back to dollar values.
- **Gradient clipping** (`max_norm=1.0`): prevents exploding gradients common in deep LSTMs.
- **Early stopping + ReduceLROnPlateau**: prevents overfitting and adapts the learning rate as training stalls.